In [0]:
path = "/Volumes/agriculture_dev/source/agriculture_dataset/agriculture_dataset.csv"

# Free-form text parameter for the CSV path
try:
    dbutils.widgets.get("SOURCE_PATH")
except Exception:
    dbutils.widgets.text("SOURCE_PATH", path)

# Dropdown parameter for load type (fixed choices)
try:
    dbutils.widgets.get("LOAD_TYPE")
except Exception:
    dbutils.widgets.dropdown("LOAD_TYPE", "FULL", ["FULL", "INCREMENTAL"], "Load Type")
SOURCE_PATH = dbutils.widgets.get("SOURCE_PATH")
LOAD_TYPE = dbutils.widgets.get("LOAD_TYPE").upper()  # FULL or INCREMENTAL

from pyspark.sql.functions import *

src = spark.read.format("csv").option("header", True).option("inferSchema", "TRUE").load(SOURCE_PATH)
expected = ["Farm_ID", "Crop_Type", "Farm_Area(acres)", "Irrigation_Type", "Fertilizer_Used(tons)", "Pesticide_Used(kg)", "Yield(tons)", "Soil_Type", "Season", "Water_Usage(cubic meters)"]

missing = sorted(set(expected) - set(src.columns))
if missing:
    raise ValueError(f"Missing required source columns: {missing}")

if LOAD_TYPE.upper() == "FULL" and spark.catalog.tableExists("agriculture_dev.bronze.agriculture_raw"):
    spark.sql(f"TRUNCATE TABLE agriculture_dev.bronze.agriculture_raw")


src_columns = src.columns

# Row-level hash of all source columns (change detection)
#withColumn("row_hash", sha2(concat_ws("||", *src_columns), 256))

# --- Incrementing batch ID ---
# A small control table remembers the last batch ID used,
# so every new run gets the next number (1, 2, 3, ...)
control_table = "agriculture_dev.bronze.batch_control"

spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {control_table} (
    last_batch_id BIGINT,
    updated_at TIMESTAMP
  )
""")

# Last used batch ID (0 on the very first run)
last_batch_id = spark.table(control_table).select(coalesce(max("last_batch_id"), lit(0))).collect()[0][0]

# Increment for this run
batch_id = int(last_batch_id) + 1

# Record this run's batch ID
spark.sql(f"INSERT INTO {control_table} VALUES ({batch_id}, current_timestamp())")

src=src.withColumn("row_hash", sha2(concat_ws("||", *src_columns), 256)).withColumn("Src_name",lit("Agriculture System")).withColumn("Ingestion_timestamp",current_timestamp()).withColumn("batch_id",lit(batch_id))

for c in src.columns:
    src = src.withColumnRenamed(c, c.replace("(", "_").replace(")", "").replace(" ", "_"))
# # df.display()
bronze_table = "agriculture_dev.bronze.agriculture_raw"

if LOAD_TYPE == "FULL":
    # Full load: rebuild bronze from scratch (table truncated above).
    # overwriteSchema=True lets the rebuild replace the existing table schema
    # (the old table was all-strings; the CSV now infers numeric types).
    src.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(bronze_table)
else:
    # Incremental load: keep history, append only new or changed rows.
    # row_hash covers all source columns, so a hash already present in
    # bronze means the row is unchanged -> skip it. A changed row gets a
    # new hash and is appended as a new version under the new batch_id.
    if spark.catalog.tableExists(bronze_table):
        existing_hashes = spark.table(bronze_table).select("row_hash")
        new_rows = src.join(existing_hashes, on="row_hash", how="left_anti")
    else:
        new_rows = src  # very first run: nothing to compare against

    new_rows_count = new_rows.count()
    print(f"Batch {batch_id}: {new_rows_count} new/changed rows to append")

    if new_rows_count > 0:
        new_rows.write.format("delta").mode("append").saveAsTable(bronze_table)
    else:
        print("No changes detected - nothing appended")

print("Bronze rows:", spark.table("agriculture_dev.bronze.agriculture_raw").count())
display(spark.table("agriculture_dev.bronze.agriculture_raw").filter(col("batch_id") == batch_id))